# Lab 4 - Silver History and SCD Type 2

Builds a history-preserving Silver table from the same durable prepared updates used by SCD Type 1.


In [0]:
%run ./lab4_00_config


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

prepared_updates = spark.table(silver_updates_table)


In [0]:
scd2_source = (
    prepared_updates
    .withColumn("valid_from", F.current_timestamp())
    .withColumn("valid_to", F.lit(None).cast("timestamp"))
    .withColumn("is_current", F.lit(True))
)

(
    scd2_source.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_scd2_stage_table)
)

scd2_stage = spark.table(silver_scd2_stage_table)

if not spark.catalog.tableExists(silver_history_table):
    (
        scd2_stage.limit(0)
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_history_table)
    )

history_target = DeltaTable.forName(spark, silver_history_table)
current_history = spark.table(silver_history_table).filter(F.col("is_current") == True)

comparison = (
    scd2_stage.alias("source")
    .join(current_history.alias("target"), F.col("source.show_id") == F.col("target.show_id"), "left")
)

changed_rows = (
    comparison
    .filter(F.col("target.show_id").isNotNull() & (F.col("source._record_hash") != F.col("target._record_hash")))
    .select("source.*")
)

new_rows = (
    comparison
    .filter(F.col("target.show_id").isNull())
    .select("source.*")
)

new_count = new_rows.count()
changed_count = changed_rows.count()

(
    history_target.alias("target")
    .merge(
        changed_rows.alias("source"),
        "target.show_id = source.show_id AND target.is_current = true"
    )
    .whenMatchedUpdate(set={
        "is_current": "false",
        "valid_to": "current_timestamp()",
        "silver_updated_at": "current_timestamp()"
    })
    .execute()
)

rows_to_insert = new_rows.unionByName(changed_rows)

if new_count + changed_count > 0:
    (
        rows_to_insert.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(silver_history_table)
    )

print("New SCD2 rows:", new_count)
print("Changed SCD2 rows:", changed_count)


In [0]:
display(
    spark.table(silver_history_table)
    .groupBy("is_current")
    .count()
)

current_duplicates = (
    spark.table(silver_history_table)
    .filter(F.col("is_current") == True)
    .groupBy("show_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Current duplicate show_id rows:", current_duplicates.count())
display(current_duplicates)
